In [6]:
import os
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq

groq_api_key = os.environ["GROQ_API_KEY"]

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=groq_api_key,
    temperature=0.2,
)

print("LLM configured for Groq successfully.")

LLM configured for Groq successfully.


In [7]:
class StudyState(TypedDict):
    topic: str
    next_agent: str
    concept: str
    example: str
    quiz: str
    feedback: str


def _normalize_supervisor_decision(text: str) -> str:
    decision = text.strip().lower()
    for option in ("concept", "example", "quiz", "feedback", "end"):
        if decision == option or decision.startswith(option):
            return option
    if "feedback" in decision:
        return "feedback"
    if "quiz" in decision:
        return "quiz"
    if "example" in decision:
        return "example"
    if "end" in decision:
        return "end"
    return "concept"


def supervisor_node(state: StudyState) -> StudyState:
    prompt = (
        "You are the supervisor in a simple teaching workflow.\n"
        f"Topic: {state['topic']}\n"
        f"Concept completed: {bool(state.get('concept', '').strip())}\n"
        f"Example completed: {bool(state.get('example', '').strip())}\n"
        f"Quiz completed: {bool(state.get('quiz', '').strip())}\n"
        f"Feedback completed: {bool(state.get('feedback', '').strip())}\n\n"
        "Decide the next action from: concept, example, quiz, feedback, end.\n"
        "Rules:\n"
        "- If concept is missing, choose concept.\n"
        "- If concept exists and example is missing, choose example.\n"
        "- If example exists and quiz is missing, choose quiz.\n"
        "- If quiz exists and feedback is missing, choose feedback.\n"
        "- If feedback exists, decide whether to repeat concept/example/quiz for another pass or end if the learner is ready.\n"
        "Return only one word.\n\n"
        "Output:"
    )
    response = llm.invoke(prompt).content
    next_agent = _normalize_supervisor_decision(response)
    return {**state, "next_agent": next_agent}


def concept_node(state: StudyState) -> StudyState:
    prompt = (
        f"Explain the concept '{state['topic']}' step-by-step for a beginner. "
        "Use simple language and short paragraphs. Focus on the core idea and how it works."
    )
    response = llm.invoke(prompt).content.strip()
    return {**state, "concept": response}


def example_node(state: StudyState) -> StudyState:
    prompt = (
        f"Based on this concept explanation:\n\n{state['concept']}\n\n"
        f"Give 2 intuitive examples of '{state['topic']}' and one short Python code example. "
        "Keep the examples simple and practical."
    )
    response = llm.invoke(prompt).content.strip()
    return {**state, "example": response}


def quiz_node(state: StudyState) -> StudyState:
    prompt = (
        f"Create 3 short quiz questions about '{state['topic']}' using this context:\n"
        f"Concept: {state['concept']}\n\nExamples: {state['example']}\n\n"
        "Keep the questions clear and beginner-friendly."
    )
    response = llm.invoke(prompt).content.strip()
    return {**state, "quiz": response}


def feedback_node(state: StudyState) -> StudyState:
    prompt = (
        f"Evaluate the quiz for the topic '{state['topic']}'.\n\n"
        f"Quiz:\n{state['quiz']}\n\n"
        "Provide a short evaluation of how well the quiz supports learning, and give suggestions for improvement or review. "
        "Return a concise response with evaluation and suggestions."
    )
    response = llm.invoke(prompt).content.strip()
    return {**state, "feedback": response}


def route_from_supervisor(state: StudyState) -> str:
    return state.get("next_agent", "concept")

In [8]:
graph = StateGraph(StudyState)

graph.add_node("supervisor", supervisor_node)
graph.add_node("concept", concept_node)
graph.add_node("example", example_node)
graph.add_node("quiz", quiz_node)
graph.add_node("feedback", feedback_node)

graph.add_edge(START, "supervisor")
graph.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "concept": "concept",
        "example": "example",
        "quiz": "quiz",
        "feedback": "feedback",
        "end": END,
    },
)
graph.add_edge("concept", "example")
graph.add_edge("example", "quiz")
graph.add_edge("quiz", "feedback")
graph.add_edge("feedback", "supervisor")

app = graph.compile()
print("LangGraph compiled successfully.")

LangGraph compiled successfully.


In [9]:
initial_state: StudyState = {
    "topic": "Teach me Binary Trees",
    "next_agent": "",
    "concept": "",
    "example": "",
    "quiz": "",
    "feedback": "",
}

print("===== WORKFLOW TRACE =====")
for step in app.stream(initial_state):
    for node_name, node_state in step.items():
        if node_name == "supervisor":
            print("\n===== SUPERVISOR DECISION =====")
            print(node_state["next_agent"])
        elif node_name == "concept":
            print("\n===== CONCEPT AGENT OUTPUT =====")
            print(node_state["concept"])
        elif node_name == "example":
            print("\n===== EXAMPLE AGENT OUTPUT =====")
            print(node_state["example"])
        elif node_name == "quiz":
            print("\n===== QUIZ AGENT OUTPUT =====")
            print(node_state["quiz"])
        elif node_name == "feedback":
            print("\n===== FEEDBACK AGENT OUTPUT =====")
            print(node_state["feedback"])

===== WORKFLOW TRACE =====

===== SUPERVISOR DECISION =====
concept

===== CONCEPT AGENT OUTPUT =====
**Introduction to Binary Trees**
A binary tree is a way to store and organize data in a computer. It's like a tree with branches, but instead of leaves, it has nodes that hold information. Each node can have up to two children, which are like smaller branches that connect to it.

**The Structure**
Imagine a tree with a trunk, branches, and leaves. In a binary tree, the trunk is called the "root" node. The root node has two children, called the "left child" and the "right child". Each of these children can have their own children, and so on. This creates a tree-like structure where each node is connected to its children and its parent.

**How it Works**
When you add data to a binary tree, it gets stored in a node. The node is then connected to its parent node, and possibly its children nodes. The tree is organized in a way that makes it easy to find specific data. For example, if you're